# Run one corrected v2 experiment
GPU required. The actual experiment logic lives in one tested script, so this notebook only chooses the run. It records stage-0 validation, uses the audited per-method replay budget, saves pair-level margins, and marks a run complete only after all stages finish.

Run the core grid first: 2 orders × 3 methods (`none`, `random`, `mfr`) × 2 seeds = 12 runs. Then run the four `random_high` controls (18 new + 3 random replay pairs). Treat `lowest_margin` as an optional secondary baseline.

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import os, sys, subprocess
if not os.path.exists('/content/mfr-dpo'):
    !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
elif subprocess.run(['git', '-C', '/content/mfr-dpo', 'status', '--porcelain', '--', 'data/v2'], capture_output=True, text=True, check=True).stdout.strip():
    print('Keeping local data/v2; skipping git pull so it is not overwritten.')
else:
    !git -C /content/mfr-dpo pull --ff-only -q
!pip install -q -r /content/mfr-dpo/requirements.txt
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/mfr-dpo'
DRIVE_DIR = '/content/drive/MyDrive/CSCI544/mfr-dpo'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Change only the values below. For replay methods, `STAGE1_FROM` should point to the matching completed v2 `none` run directory for the same order and seed. Never point it to the old pilot.

In [14]:
ORDER_ID = 1                # 1 or 2
METHOD = 'none'             # none | random | random_high | mfr | lowest_margin
SEED = 0                    # 0 or 1
START_STAGE = 1             # set 2 or 3 only when resuming this same run
STAGE1_FROM = None          # e.g. f'{DRIVE_DIR}/runs/v2_o2_none_s0'
REFERENCE_CACHE = f'{DRIVE_DIR}/cache/reference_v2.csv'

In [15]:
command = [sys.executable, '-u', f'{REPO}/scripts/run_experiment.py', '--drive-dir', DRIVE_DIR,
           '--order', str(ORDER_ID), '--method', METHOD, '--seed', str(SEED),
           '--start-stage', str(START_STAGE)]
if STAGE1_FROM:
    command += ['--stage1-from', STAGE1_FROM]
if os.path.exists(REFERENCE_CACHE):
    command += ['--reference-cache', REFERENCE_CACHE]
print(' '.join(command))
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(4096)
    if not chunk:
        break
    print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Experiment stopped with exit code {return_code}; see the specific error above.')

/usr/bin/python3 -u /content/mfr-dpo/scripts/run_experiment.py --drive-dir /content/drive/MyDrive/CSCI544/mfr-dpo --order 1 --method none --seed 0 --start-stage 1 --reference-cache /content/drive/MyDrive/CSCI544/mfr-dpo/cache/reference_v2.csv
Starting v2_o1_none_s0: order=helpful -> safe -> quality, batch=18 new + 0 replay
Loading and verifying the frozen data...
Loading Qwen/Qwen2.5-1.5B-Instruct and the starting adapter...
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 315.03it/s]
Model loaded.
cache canary: cached reference: 100%|██████████| 8/8 [00:02<00:00,  2.99batch/s]
Reference cache canary passed: {'sample_size': 32, 'batch_size': 4, 'sampling': 'aligned_length_batches', 'max_abs_margin_difference': 1.2890159784006983e-07, 'mean_abs_margin_difference': 2.648783314559977e-08, 'tolerance': 0.001}
stage 0: quality validation: 100%|██████████| 50/50 [00:15<00:00,  3.18batch/s]

Stage 1/3: training on helpful
stage 1: helpful: 100%|██████████| 112/112 [10:49<00:00,  5.80s/

## MFR-DPO Experiment Checklist

### Required experiments — 16 runs

| Sequence | Order | Seed | Run order | Run name | Method | Stage-1 source | Status |
|---:|---:|---:|---:|---|---|---|:---:|
| 1 | 1 | 0 | 1 | `v2_o1_none_s0` | `none` | `None` | ✅ |
| 1 | 1 | 0 | 2 | `v2_o1_random_s0` | `random` | `v2_o1_none_s0` | ✅ |
| 1 | 1 | 0 | 3 | `v2_o1_mfr_s0` | `mfr` | `v2_o1_none_s0` | ✅ |
| 1 | 1 | 0 | 4 | `v2_o1_random_high_s0` | `random_high` | `v2_o1_none_s0` | ✅ |
| 2 | 1 | 1 | 1 | `v2_o1_none_s1` | `none` | `None` | ⬜ |
| 2 | 1 | 1 | 2 | `v2_o1_mfr_s1` | `mfr` | `v2_o1_none_s1` | ⬜ |
| 2 | 1 | 1 | 3 | `v2_o1_random_high_s1` | `random_high` | `v2_o1_none_s1` | ⬜ |
| 2 | 1 | 1 | 4 | `v2_o1_random_s1` | `random` | `v2_o1_none_s1` | ⬜ |
| 3 | 2 | 0 | 1 | `v2_o2_none_s0` | `none` | `None` | ⬜ |
| 3 | 2 | 0 | 2 | `v2_o2_random_high_s0` | `random_high` | `v2_o2_none_s0` | ⬜ |
| 3 | 2 | 0 | 3 | `v2_o2_random_s0` | `random` | `v2_o2_none_s0` | ⬜ |
| 3 | 2 | 0 | 4 | `v2_o2_mfr_s0` | `mfr` | `v2_o2_none_s0` | ⬜ |
| 4 | 2 | 1 | 1 | `v2_o2_none_s1` | `none` | `None` | ⬜ |
| 4 | 2 | 1 | 2 | `v2_o2_random_s1` | `random` | `v2_o2_none_s1` | ⬜ |
| 4 | 2 | 1 | 3 | `v2_o2_mfr_s1` | `mfr` | `v2_o2_none_s1` | ⬜ |
| 4 | 2 | 1 | 4 | `v2_o2_random_high_s1` | `random_high` | `v2_o2_none_s1` | ⬜ |

### Optional experiments — Lowest-margin baseline

Run these only after completing the 16 required runs.

| Order | Seed | Run name | Method | Stage-1 source | Status |
|---:|---:|---|---|---|:---:|
| 1 | 0 | `v2_o1_lowest_margin_s0` | `lowest_margin` | `v2_o1_none_s0` | ✅ |
| 1 | 1 | `v2_o1_lowest_margin_s1` | `lowest_margin` | `v2_o1_none_s1` | ⬜ |
| 2 | 0 | `v2_o2_lowest_margin_s0` | `lowest_margin` | `v2_o2_none_s0` | ⬜ |
| 2 | 1 | `v2_o2_lowest_margin_s1` | `lowest_margin` | `v2_o2_none_s1` | ⬜ |

### Analysis and evaluation

| Step | When to run | Status |
|---|---|:---:|
| Notebook 07 — compare validation results | After required runs | ⬜ |
| Notebook 08 — error analysis | After notebook 07 | ⬜ |
| Freeze model-selection decision | Before opening test results | ⬜ |
| Notebook 09 — locked test evaluation | After selection is frozen | ⬜ |
| Notebook 10 — generation evaluation | After final test evaluation | ⬜ |
| IFEval evaluation | Optional secondary evaluation | ⬜ |
| Notebook 11 — blinded human review | After generation | ⬜ |

In [24]:
ORDER_ID = 1                # 1 or 2
METHOD = 'random'             # none | random | random_high | mfr | lowest_margin
SEED = 0                    # 0 or 1
START_STAGE = 1             # set 2 or 3 only when resuming this same run
STAGE1_FROM = f"{DRIVE_DIR}/runs/v2_o1_none_s0"
REFERENCE_CACHE = f"{DRIVE_DIR}/cache/reference_v2.csv"

In [25]:
command = [sys.executable, '-u', f'{REPO}/scripts/run_experiment.py', '--drive-dir', DRIVE_DIR,
           '--order', str(ORDER_ID), '--method', METHOD, '--seed', str(SEED),
           '--start-stage', str(START_STAGE)]
if STAGE1_FROM:
    command += ['--stage1-from', STAGE1_FROM]
if os.path.exists(REFERENCE_CACHE):
    command += ['--reference-cache', REFERENCE_CACHE]
print(' '.join(command))
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(4096)
    if not chunk:
        break
    print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Experiment stopped with exit code {return_code}; see the specific error above.')

/usr/bin/python3 -u /content/mfr-dpo/scripts/run_experiment.py --drive-dir /content/drive/MyDrive/CSCI544/mfr-dpo --order 1 --method random --seed 0 --start-stage 1 --stage1-from /content/drive/MyDrive/CSCI544/mfr-dpo/runs/v2_o1_none_s0 --reference-cache /content/drive/MyDrive/CSCI544/mfr-dpo/cache/reference_v2.csv
Starting v2_o1_random_s0: order=helpful -> safe -> quality, batch=18 new + 2 replay
Loading and verifying the frozen data...
Loading Qwen/Qwen2.5-1.5B-Instruct and the starting adapter...
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 317.74it/s]
Model loaded.
cache canary: cached reference: 100%|██████████| 8/8 [00:02<00:00,  2.99batch/s]
Reference cache canary passed: {'sample_size': 32, 'batch_size': 4, 'sampling': 'aligned_length_batches', 'max_abs_margin_difference': 1.0896474123001099e-07, 'mean_abs_margin_difference': 2.2753283701604232e-08, 'tolerance': 0.001}

Stage 2/3: training on safe
stage 2: safe: 100%|██████████| 112/112 [10:43<00:00,  5.75s/step, acc

In [26]:
ORDER_ID = 1                # 1 or 2
METHOD = 'mfr'             # none | random | random_high | mfr | lowest_margin
SEED = 0                    # 0 or 1
START_STAGE = 1             # set 2 or 3 only when resuming this same run
STAGE1_FROM = f"{DRIVE_DIR}/runs/v2_o1_none_s0"
REFERENCE_CACHE = f"{DRIVE_DIR}/cache/reference_v2.csv"

In [27]:
command = [sys.executable, '-u', f'{REPO}/scripts/run_experiment.py', '--drive-dir', DRIVE_DIR,
           '--order', str(ORDER_ID), '--method', METHOD, '--seed', str(SEED),
           '--start-stage', str(START_STAGE)]
if STAGE1_FROM:
    command += ['--stage1-from', STAGE1_FROM]
if os.path.exists(REFERENCE_CACHE):
    command += ['--reference-cache', REFERENCE_CACHE]
print(' '.join(command))
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(4096)
    if not chunk:
        break
    print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Experiment stopped with exit code {return_code}; see the specific error above.')

/usr/bin/python3 -u /content/mfr-dpo/scripts/run_experiment.py --drive-dir /content/drive/MyDrive/CSCI544/mfr-dpo --order 1 --method mfr --seed 0 --start-stage 1 --stage1-from /content/drive/MyDrive/CSCI544/mfr-dpo/runs/v2_o1_none_s0 --reference-cache /content/drive/MyDrive/CSCI544/mfr-dpo/cache/reference_v2.csv
Starting v2_o1_mfr_s0: order=helpful -> safe -> quality, batch=18 new + 2 replay
Loading and verifying the frozen data...
Loading Qwen/Qwen2.5-1.5B-Instruct and the starting adapter...
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 315.30it/s]
Model loaded.
cache canary: cached reference: 100%|██████████| 8/8 [00:02<00:00,  2.99batch/s]
Reference cache canary passed: {'sample_size': 32, 'batch_size': 4, 'sampling': 'aligned_length_batches', 'max_abs_margin_difference': 1.0291114449501038e-07, 'mean_abs_margin_difference': 2.18842615140602e-08, 'tolerance': 0.001}

Stage 2/3: training on safe
stage 2: safe: 100%|██████████| 112/112 [13:03<00:00,  7.00s/step, accuracy=10

In [28]:
ORDER_ID = 1                # 1 or 2
METHOD = 'random_high'             # none | random | random_high | mfr | lowest_margin
SEED = 0                    # 0 or 1
START_STAGE = 1             # set 2 or 3 only when resuming this same run
STAGE1_FROM = f"{DRIVE_DIR}/runs/v2_o1_none_s0"
REFERENCE_CACHE = f"{DRIVE_DIR}/cache/reference_v2.csv"

In [29]:
command = [sys.executable, '-u', f'{REPO}/scripts/run_experiment.py', '--drive-dir', DRIVE_DIR,
           '--order', str(ORDER_ID), '--method', METHOD, '--seed', str(SEED),
           '--start-stage', str(START_STAGE)]
if STAGE1_FROM:
    command += ['--stage1-from', STAGE1_FROM]
if os.path.exists(REFERENCE_CACHE):
    command += ['--reference-cache', REFERENCE_CACHE]
print(' '.join(command))
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(4096)
    if not chunk:
        break
    print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Experiment stopped with exit code {return_code}; see the specific error above.')

/usr/bin/python3 -u /content/mfr-dpo/scripts/run_experiment.py --drive-dir /content/drive/MyDrive/CSCI544/mfr-dpo --order 1 --method random_high --seed 0 --start-stage 1 --stage1-from /content/drive/MyDrive/CSCI544/mfr-dpo/runs/v2_o1_none_s0 --reference-cache /content/drive/MyDrive/CSCI544/mfr-dpo/cache/reference_v2.csv
Starting v2_o1_random_high_s0: order=helpful -> safe -> quality, batch=18 new + 3 replay
Loading and verifying the frozen data...
Loading Qwen/Qwen2.5-1.5B-Instruct and the starting adapter...
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 321.34it/s]
Model loaded.
cache canary: cached reference: 100%|██████████| 8/8 [00:02<00:00,  2.98batch/s]
Reference cache canary passed: {'sample_size': 32, 'batch_size': 4, 'sampling': 'aligned_length_batches', 'max_abs_margin_difference': 1.1478550732135773e-07, 'mean_abs_margin_difference': 2.0816059986827895e-08, 'tolerance': 0.001}

Stage 2/3: training on safe
stage 2: safe: 100%|██████████| 112/112 [11:37<00:00,  6.23s

In [30]:
ORDER_ID = 1                # 1 or 2
METHOD = 'lowest_margin'             # none | random | random_high | mfr | lowest_margin
SEED = 0                    # 0 or 1
START_STAGE = 1             # set 2 or 3 only when resuming this same run
STAGE1_FROM = f"{DRIVE_DIR}/runs/v2_o1_none_s0"
REFERENCE_CACHE = f"{DRIVE_DIR}/cache/reference_v2.csv"

In [31]:
command = [sys.executable, '-u', f'{REPO}/scripts/run_experiment.py', '--drive-dir', DRIVE_DIR,
           '--order', str(ORDER_ID), '--method', METHOD, '--seed', str(SEED),
           '--start-stage', str(START_STAGE)]
if STAGE1_FROM:
    command += ['--stage1-from', STAGE1_FROM]
if os.path.exists(REFERENCE_CACHE):
    command += ['--reference-cache', REFERENCE_CACHE]
print(' '.join(command))
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(4096)
    if not chunk:
        break
    print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
return_code = process.wait()
if return_code:
    raise RuntimeError(f'Experiment stopped with exit code {return_code}; see the specific error above.')

/usr/bin/python3 -u /content/mfr-dpo/scripts/run_experiment.py --drive-dir /content/drive/MyDrive/CSCI544/mfr-dpo --order 1 --method lowest_margin --seed 0 --start-stage 1 --stage1-from /content/drive/MyDrive/CSCI544/mfr-dpo/runs/v2_o1_none_s0 --reference-cache /content/drive/MyDrive/CSCI544/mfr-dpo/cache/reference_v2.csv
Starting v2_o1_lowest_margin_s0: order=helpful -> safe -> quality, batch=18 new + 2 replay
Loading and verifying the frozen data...
Loading Qwen/Qwen2.5-1.5B-Instruct and the starting adapter...
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 322.28it/s]
Model loaded.
cache canary: cached reference: 100%|██████████| 8/8 [00:02<00:00,  2.98batch/s]
Reference cache canary passed: {'sample_size': 32, 'batch_size': 4, 'sampling': 'aligned_length_batches', 'max_abs_margin_difference': 1.2526288628578186e-07, 'mean_abs_margin_difference': 2.1295363694662228e-08, 'tolerance': 0.001}

Stage 2/3: training on safe
stage 2: safe: 100%|██████████| 112/112 [13:21<00:00,  7